# Group 3: Naive Bayes Spam Classification (Colab Ready)

This notebook is a **single cohesive presentation** for your AI Modeling course practical. It follows the required pipeline from Step 1 to Step 8 and adds an interactive demo for real-time spam prediction.

---

## Step 1: Clarify The Task

**Problem type:** Binary classification (Spam = 1, Not Spam/Ham = 0)

**Goal:** Build a Naive Bayes model that classifies text messages as spam or not spam.

**Success metrics:**
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

**Constraints:**
- Must be explainable for classroom presentation
- Must run on Google Colab
- Should show both theory and practical inference

In [ ]:
# Colab-friendly setup (silent install).
!pip -q install -U pandas numpy scikit-learn matplotlib seaborn datasets ipywidgets wordcloud

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datasets import load_dataset
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

warnings.filterwarnings("ignore")
SEED = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 140)
print("Libraries loaded successfully.")

## Step 2: Data Collection

We first try to load an SMS spam dataset from Hugging Face. If a particular dataset name changes or is unavailable, we automatically fall back to another source.

This makes the notebook robust for classroom demos on Colab.

In [ ]:
def normalize_labels(label_series):
    """Convert many possible label formats into binary: spam=1, ham=0."""
    s = label_series.astype(str).str.lower().str.strip()

    if s.isin(["0", "1"]).all():
        return s.astype(int)

    def map_label(x):
        if x in {"ham", "not spam", "legit", "legitimate", "0", "no", "false"}:
            return 0
        if x in {"spam", "1", "yes", "true", "junk"}:
            return 1
        if "ham" in x:
            return 0
        if ("spam" in x) and ("not" not in x):
            return 1
        return np.nan

    y = s.apply(map_label)
    return y


def load_spam_dataframe():
    # (dataset_name, split) candidates
    hf_candidates = [
        ("ucirvine/sms_spam", "train"),
        ("sms_spam", "train"),
        ("SetFit/sms_spam", "train")
    ]

    for ds_name, split_name in hf_candidates:
        try:
            ds = load_dataset(ds_name, split=split_name)
            df = ds.to_pandas()

            text_col = next((c for c in ["text", "sms", "message", "body", "content"] if c in df.columns), None)
            label_col = next((c for c in ["label", "spam", "target", "category", "class"] if c in df.columns), None)

            if text_col and label_col:
                out = df[[text_col, label_col]].copy()
                out.columns = ["text", "label_raw"]
                out["label"] = normalize_labels(out["label_raw"])
                out = out.dropna(subset=["text", "label"]).copy()
                out["text"] = out["text"].astype(str).str.strip()
                out = out[out["text"].str.len() > 0]
                out["label"] = out["label"].astype(int)
                print(f"Loaded dataset from Hugging Face: {ds_name}")
                return out[["text", "label"]]
        except Exception as e:
            print(f"Could not load {ds_name}: {type(e).__name__}")

    # Fallback: OpenML
    print("Falling back to OpenML 'sms_spam' dataset...")
    openml_data = fetch_openml(name="sms_spam", version=1, as_frame=True)
    df = openml_data.frame.copy()

    text_col = next((c for c in ["text", "sms", "message", "body"] if c in df.columns), df.columns[0])
    label_col = next((c for c in ["class", "label", "target", "spam"] if c in df.columns), df.columns[-1])

    out = df[[text_col, label_col]].copy()
    out.columns = ["text", "label_raw"]
    out["label"] = normalize_labels(out["label_raw"])
    out = out.dropna(subset=["text", "label"]).copy()
    out["text"] = out["text"].astype(str).str.strip()
    out = out[out["text"].str.len() > 0]
    out["label"] = out["label"].astype(int)
    return out[["text", "label"]]


df = load_spam_dataframe()
print(f"Dataset shape: {df.shape}")
display(df.head())
print(df['label'].value_counts().rename(index={0: 'ham', 1: 'spam'}))

## Step 3: Exploratory Data Analysis (EDA)

We summarize key statistics, inspect class balance, and visualize text length distributions (a useful signal in spam detection).

In [ ]:
df['char_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("Summary statistics:")
display(df[['char_length', 'word_count']].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='label', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution (0 = Ham, 1 = Spam)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

sns.histplot(data=df, x='char_length', hue='label', bins=50, kde=True, ax=axes[1], palette='Set1')
axes[1].set_title('Message Length Distribution')
axes[1].set_xlabel('Character Length')

plt.tight_layout()
plt.show()

## Step 4: Data Cleaning & Preprocessing

We clean text by lowercasing, removing URLs, stripping punctuation/non-letters, and normalizing spaces.

For features, we use **TF-IDF** so Naive Bayes can learn token importance efficiently.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)

display(df[['text', 'clean_text', 'label']].head(8))
print(f"Rows after cleaning: {len(df)}")

## Step 5: Train / Validation / Test Split

We use a stratified split to preserve class balance:
- Train: 70%
- Validation: 15%
- Test: 15%

In [ ]:
X = df['clean_text']
y = df['label']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(0.15 / 0.85), random_state=SEED, stratify=y_temp
)

print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

## Step 6: Model Selection (Naive Bayes Candidates)

We compare two Naive Bayes variants using grid search:
- MultinomialNB
- ComplementNB

Both are popular for text classification tasks.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('nb', MultinomialNB())
])

param_grid = [
    {
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'tfidf__min_df': [1, 2, 3],
        'nb': [MultinomialNB()],
        'nb__alpha': [0.1, 0.5, 1.0]
    },
    {
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'tfidf__min_df': [1, 2, 3],
        'nb': [ComplementNB()],
        'nb__alpha': [0.1, 0.5, 1.0]
    }
]

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print('Best parameters:')
print(grid.best_params_)
print(f"Best CV F1-score: {grid.best_score_:.4f}")

## Step 7: Train The Model

GridSearchCV already trained all candidate models and selected the best one.

Now we check validation performance before final testing.

In [ ]:
def evaluate_split(model, X_split, y_split, split_name='Split'):
    pred = model.predict(X_split)
    prob = model.predict_proba(X_split)[:, 1]

    metrics = {
        'split': split_name,
        'accuracy': accuracy_score(y_split, pred),
        'precision': precision_score(y_split, pred, zero_division=0),
        'recall': recall_score(y_split, pred, zero_division=0),
        'f1': f1_score(y_split, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_split, prob)
    }
    return metrics, pred, prob

val_metrics, y_val_pred, y_val_prob = evaluate_split(best_model, X_val, y_val, split_name='Validation')
display(pd.DataFrame([val_metrics]).set_index('split').round(4))
print('Validation classification report:')
print(classification_report(y_val, y_val_pred, digits=4))

## Step 8: Model Evaluation On Test Set

We now evaluate the selected model on unseen test data with relevant metrics and visualizations.

In [ ]:
test_metrics, y_test_pred, y_test_prob = evaluate_split(best_model, X_test, y_test, split_name='Test')
display(pd.DataFrame([test_metrics]).set_index('split').round(4))

print('Test classification report:')
print(classification_report(y_test, y_test_pred, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix (Test)')

RocCurveDisplay.from_predictions(y_test, y_test_prob, ax=axes[1])
axes[1].set_title('ROC Curve (Test)')

plt.tight_layout()
plt.show()

## Naive Bayes Theory (For Presentation)

Bayes theorem for classification:

$$P(y \mid x) = \frac{P(x \mid y)P(y)}{P(x)}$$

For text, Naive Bayes assumes token independence given class and works with log-probabilities:

$$\log P(y \mid x) \propto \log P(y) + \sum_i x_i \log P(w_i \mid y)$$

where:
- $P(y)$ is the class prior (spam/ham base rate)
- $P(w_i \mid y)$ is token likelihood for class $y$
- $x_i$ is token count/weight in the message

We also apply Laplace smoothing in Naive Bayes to avoid zero probabilities.

In [ ]:
# Show top indicative words for spam vs ham based on learned log-probabilities.
vectorizer = best_model.named_steps['tfidf']
nb_model = best_model.named_steps['nb']
feature_names = np.array(vectorizer.get_feature_names_out())

class_order = list(nb_model.classes_)
ham_idx = class_order.index(0)
spam_idx = class_order.index(1)

delta = nb_model.feature_log_prob_[spam_idx] - nb_model.feature_log_prob_[ham_idx]

top_spam_idx = np.argsort(delta)[-20:][::-1]
top_ham_idx = np.argsort(delta)[:20]

top_spam_words = pd.DataFrame({
    'token': feature_names[top_spam_idx],
    'spam_minus_ham_log_prob': delta[top_spam_idx]
})

top_ham_words = pd.DataFrame({
    'token': feature_names[top_ham_idx],
    'spam_minus_ham_log_prob': delta[top_ham_idx]
})

print('Top spam-indicative tokens:')
display(top_spam_words)
print('Top ham-indicative tokens:')
display(top_ham_words)

In [ ]:
def explain_naive_bayes_prediction(message, model, top_n=10):
    cleaned = clean_text(message)
    vec = model.named_steps['tfidf']
    nb = model.named_steps['nb']

    x = vec.transform([cleaned]).toarray().ravel()
    nonzero_idx = np.where(x > 0)[0]

    class_order = list(nb.classes_)
    ham_i = class_order.index(0)
    spam_i = class_order.index(1)

    spam_score = nb.class_log_prior_[spam_i] + np.dot(x, nb.feature_log_prob_[spam_i])
    ham_score = nb.class_log_prior_[ham_i] + np.dot(x, nb.feature_log_prob_[ham_i])

    pred = 1 if spam_score > ham_score else 0
    proba = model.predict_proba([cleaned])[0, 1]

    print(f"Message: {message}")
    print(f"Cleaned : {cleaned}")
    print(f"Log score (ham) : {ham_score:.4f}")
    print(f"Log score (spam): {spam_score:.4f}")
    print(f"Predicted class : {'SPAM' if pred == 1 else 'HAM'}")
    print(f"Spam probability: {proba:.4f}")

    if len(nonzero_idx) == 0:
        print('No known tokens from vocabulary were found in this message.')
        return

    feats = np.array(vec.get_feature_names_out())[nonzero_idx]
    contrib = x[nonzero_idx] * (nb.feature_log_prob_[spam_i, nonzero_idx] - nb.feature_log_prob_[ham_i, nonzero_idx])

    contrib_df = pd.DataFrame({
        'token': feats,
        'contribution_to_spam_minus_ham': contrib
    }).sort_values('contribution_to_spam_minus_ham', ascending=False)

    print(f"\nTop {top_n} token contributions toward SPAM:")
    display(contrib_df.head(top_n))


example_message = "Congratulations! You have won a free ticket. Reply now to claim your prize."
explain_naive_bayes_prediction(example_message, best_model, top_n=12)

## Live Inference Demo (Real-Time Classification)

Type any message in the input box and the notebook predicts whether it is spam or ham.

This is useful for your final in-class demonstration.

In [ ]:
from ipywidgets import interact, Textarea

def classify_message_live(message):
    cleaned = clean_text(message)
    pred = best_model.predict([cleaned])[0]
    prob = best_model.predict_proba([cleaned])[0, 1]
    print(f"Input text: {message}")
    print(f"Prediction: {'SPAM' if pred == 1 else 'HAM'}")
    print(f"Spam probability: {prob:.4f}")

interact(
    classify_message_live,
    message=Textarea(
        value='Hello, are we still meeting at 4pm today?',
        placeholder='Type a message to classify...',
        description='Message:',
        layout={'width': '700px', 'height': '120px'}
    )
)

## Final Notes For Presentation

1. Explain why this is a **classification** task (spam vs ham).
2. Present data source and class balance.
3. Show cleaning and TF-IDF preprocessing.
4. Explain Bayes formula and log-probability idea.
5. Present validation/test metrics and confusion matrix.
6. End with the live inference demo to prove practical utility.

You can now upload this notebook directly to Google Colab and run top-to-bottom.